# NB13 — Method extensions and label-free baselines

**Purpose.** Second revision round. Uses only files already on Drive (no BigQuery). (A) label-free baselines: prevalence correction from each site's mortality rate, and Saerens EM label-shift correction from unlabeled scores; (B) covariate-informed empirical Bayes (Fay-Herriot prior mean from hospital descriptors); (C) semi-supervised hierarchical recalibration (labels + unlabeled EM estimate + covariate prior); (D) bivariate intercept-slope empirical Bayes; (E) summary with chance ceilings, tau CIs and Holm tests; (F) leave-region-out for the new methods; (G) label budget from 0 to 400 labels with squared-error outputs; (H) inputs for the closed-form risk analysis; (I) MLP base model; (J) second outcome with the new methods.

**Research question.** Revision round 2

| | |
|---|---|
| **Inputs** | `models/pred_eicu.parquet`, `models/pred_eval_strategies_rev.parquet`, analysis and raw parquet |
| **Outputs** | `results/NB13/*.csv`, `results/NB13_results.zip` |
| **Free-Colab runtime** | 30-60 min (checkpointed) |

**How to run:** `Runtime → Run all`. If Colab disconnects, just run all again — finished steps are cached.
At the end, download the zip printed in the last cell into your local `Plan A/v4/results/` folder.

In [ ]:
# ==== Setup (identical in every notebook) ====
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
BASE = os.environ['TRANSPORT_BASE']
sys.path.insert(0, os.path.join(BASE, 'code'))
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
importlib.reload(tu)
warnings.filterwarnings('ignore')
tu.set_style()
OUT = tu.init_notebook('NB13')

### 0. Load

In [ ]:
import json, lightgbm as lgb
from scipy import stats
from scipy.stats import wilcoxon
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
m, e = tu.load_analysis()
pred = pd.read_parquet(os.path.join(tu.MODELS, 'pred_eicu.parquet'))
evr = pd.read_parquet(os.path.join(tu.MODELS, 'pred_eval_strategies_rev.parquet'))
bp = json.load(open(os.path.join(tu.MODELS, 'lgbm_best_params.json')))
FEATS = tu.feature_set('V1')
hy = pd.read_csv(os.path.join(tu.RESULTS, 'NB03', 'hospital_year_counts.csv'))
ELIG = sorted(set(hy.loc[hy['eligible_temporal'], 'hospital_id']))
upd = pred[pred['split'] == 'update'].reset_index(drop=True)
ev = pred[pred['split'] == 'eval'].reset_index(drop=True)
PI_SRC = float(m.loc[m['period'] == 'dev', 'mortality'].mean())   # source (development) prevalence
B = tu.n_boot(1000); R_SIM = 100 if tu.FAST else 1000; R_LB = 3 if tu.FAST else 50; B_EM = 5 if tu.FAST else 30
L = dict(tu.STRATEGY_LABELS)
L.update({'S3_eb_loho': 'EB, leave-hospital-out prior', 'S3_eb_noglobal_loho': 'EB without global step (LOHO prior)',
          'S3_glmm': 'One-stage random-intercept GLMM', 'S3_glmm_loho': 'One-stage GLMM, leave-hospital-out',
          'S6_prev': 'Prevalence correction (site mortality rate)', 'S6_em': 'EM label-shift correction (no labels)',
          'S3_cov': 'Covariate-informed EB', 'S3_cov_loho': 'Covariate-informed EB, LOHO',
          'S3_ss': 'Semi-supervised covariate EB (labels + EM), LOHO', 'S3_slope': 'Bivariate intercept-slope EB'})
# hospital descriptors (static) + label-free 2014 summaries (mean predicted risk, mean age)
hd = e.groupby('hospital_id').agg(numbedscategory=('numbedscategory', 'first'), teachingstatus=('teachingstatus', 'first'),
                                  region=('region', 'first')).reset_index()
u14 = upd.groupby('hospital_id').agg(mean_p=('p_lgbm', 'mean'), mean_age=('age', 'mean'), n14=('mortality', 'size'),
                                     rate14=('mortality', 'mean')).reset_index()
hd = hd.merge(u14, on='hospital_id', how='left')
print('source prevalence', round(PI_SRC, 4), '| eligible', len(ELIG), '| update hospitals', upd.hospital_id.nunique())

### Helpers

In [ ]:
def by_hosp(df, pcol, ycol='mortality'):
    return {h: (d[pcol].values.astype(float), d[ycol].values.astype(float)) for h, d in df.groupby('hospital_id')}

def em_prior(q, pi_s, iters=500):
    """Saerens-Latinne-Decaestecker EM: prior of the target site under pure label shift (no labels)."""
    q = np.clip(np.asarray(q, float), 1e-6, 1 - 1e-6); pi = pi_s
    for _ in range(iters):
        w1 = q * pi / pi_s; w0 = (1 - q) * (1 - pi) / (1 - pi_s)
        new = float(np.mean(w1 / (w1 + w0)))
        if abs(new - pi) < 1e-10:
            break
        pi = new
    return pi

def em_offset(q, pi_s, nboot=0, rng=None):
    """Label-free logit offset logit(pi_t) - logit(pi_s) and its bootstrap SE."""
    lg = lambda x: np.log(x / (1 - x))
    off = lg(em_prior(q, pi_s)) - lg(pi_s)
    if not nboot:
        return off, np.nan
    rng = rng or np.random.default_rng(0)
    bs = [lg(em_prior(q[rng.integers(0, len(q), len(q))], pi_s)) - lg(pi_s) for _ in range(nboot)]
    return off, float(np.std(bs, ddof=1))

def zmatrix(hids, use_region=True):
    d = hd.set_index('hospital_id').reindex(hids)
    Z = pd.DataFrame(index=d.index)
    bc = d['numbedscategory'].fillna('Unknown').astype(str)
    for c in ['< 100', '<100', '250 - 499', '>= 500', 'Unknown']:
        if (bc == c).any():
            Z['beds_' + c] = (bc == c).astype(float)
    Z['teaching'] = (d['teachingstatus'].astype(str).str.lower() == 'true').astype(float)
    if use_region:
        rg = d['region'].fillna('Unknown').astype(str)
        for c in ['Northeast', 'South', 'West', 'Unknown']:
            Z['region_' + c] = (rg == c).astype(float)
    Z['mean_risk_x10'] = 10 * d['mean_p'].fillna(d['mean_p'].mean())
    Z['mean_age_10y'] = d['mean_age'].fillna(d['mean_age'].mean()) / 10
    Z.insert(0, 'const', 1.0)
    return Z

def meta_reg_mom(a, s, Z):
    """Mixed-effects meta-regression: method-of-moments residual tau^2 and WLS coefficients."""
    a, v, X = np.asarray(a, float), np.asarray(s, float) ** 2, np.asarray(Z, float)
    keep = X.std(0) > 0; keep[0] = True; X = X[:, keep]
    W = np.diag(1 / v)
    XtWX = X.T @ W @ X
    P = W - W @ X @ np.linalg.pinv(XtWX) @ X.T @ W
    tau2 = max(0.0, (float(a @ P @ a) - (len(a) - X.shape[1])) / np.trace(P))
    W2 = np.diag(1 / (v + tau2))
    g = np.linalg.pinv(X.T @ W2 @ X) @ X.T @ W2 @ a
    return g, tau2, keep

EMALL = None
def network_fit(G, hosp, global_step=True, use_region=True, with_em=False, pi_net=None, rng=None):
    """Fit everything the network supplies from the 2014 data of `hosp`: global logistic recalibration,
    per-site offsets (a, s), plain EB (mu, tau2), covariate EB (gamma, tau2_res), and, optionally, the
    EM-bias variance sigma_b^2 for the semi-supervised method."""
    if global_step:
        p = np.concatenate([G[h][0] for h in hosp]); y = np.concatenate([G[h][1] for h in hosp])
        ag, bg = tu.fit_logistic(p, y)
    else:
        ag, bg = 0.0, 1.0
    rows = []
    for h in hosp:
        q = tu.apply_logistic(G[h][0], ag, bg)
        a, s = tu.fit_intercept(q, G[h][1])
        rows.append((h, a, s))
    t = pd.DataFrame(rows, columns=['h', 'a', 's']).set_index('h')
    ma = tu.meta_analysis(t['a'], t['s'], hksj=False)
    Z = zmatrix(list(t.index), use_region)
    g, tau2r, keep = meta_reg_mom(t['a'].values, t['s'].values, Z)
    out = dict(ag=ag, bg=bg, mu=ma['mu'], tau2=ma['tau2'], gamma=g, keep=keep, tau2_res=tau2r, use_region=use_region,
               zcols=list(Z.columns))
    if with_em:
        if pi_net is None:
            yy = np.concatenate([G[h][1] for h in hosp]); pi_net = float(yy.mean())
        if EMALL is None:      # first (all-network) call: compute every site's unlabeled EM offset once
            e_ = []
            for h in hosp:
                q = tu.apply_logistic(G[h][0], ag, bg)
                e_.append(em_offset(q, pi_net, nboot=B_EM, rng=rng))
            e_ = np.array(e_)
        else:                  # later calls reuse the precomputed unlabeled offsets (label-free)
            e_ = np.array([EMALL[h] for h in hosp])
        diff = e_[:, 0] - t['a'].values
        sb2 = max(0.0, float(np.var(diff, ddof=1) - np.mean(e_[:, 1] ** 2) - np.mean(t['s'].values ** 2)))
        out.update(pi_net=pi_net, sigma_b2=sb2)
    return out

def cov_mean(fit, h):
    Z = zmatrix([h], fit['use_region'])
    Z = Z.reindex(columns=fit['zcols'], fill_value=0.0).values[:, fit['keep']]
    return float((Z @ fit['gamma'])[0])

def posterior(m_, t2, a=None, s=None, eoff=None, ese=None, sb2=None):
    t2 = max(float(t2), 1e-8)
    prec, num = 1 / t2, m_ / t2
    if a is not None:
        prec += 1 / s ** 2; num += a / s ** 2
    if eoff is not None:
        ve = ese ** 2 + sb2
        prec += 1 / ve; num += eoff / ve
    return num / prec

def hosp_summary(df, pcol, label, hosp, ycol='mortality'):
    ph = tu.per_hospital(df[df['hospital_id'].isin(hosp)], pcol, ycol=ycol, min_n=0, min_events=0, full=False)
    ma = tu.meta_analysis(ph['ln_oe'], ph['ln_oe_se'])
    return ph, dict(strategy=label, hospitals=len(ph), pct_acceptable=100 * ph['oe'].between(*tu.OE_ACCEPT).mean(),
                    median_abs_ln_oe=ph['ln_oe'].abs().median(), mean_sq_ln_oe=(ph['ln_oe'] ** 2).mean(),
                    tau=ma['tau'], pooled_oe_re=np.exp(ma['mu']), pi_lo=np.exp(ma['pi_lo']), pi_hi=np.exp(ma['pi_hi']))

def chance_ceiling(df, pcol, hosp, R=R_SIM, seed=7):
    rng = np.random.default_rng(seed)
    g = {h: d[pcol].values for h, d in df[df['hospital_id'].isin(hosp)].groupby('hospital_id')}
    H = list(g); O = np.zeros((R, len(H))); E = np.array([g[h].sum() for h in H])
    for j, h in enumerate(H):
        p = g[h]
        for r0 in range(0, R, 250):
            r1 = min(R, r0 + 250); O[r0:r1, j] = (rng.random((r1 - r0, len(p))) < p).sum(1)
    oe = O / E
    inband = 100 * ((oe >= tu.OE_ACCEPT[0]) & (oe <= tu.OE_ACCEPT[1])).mean(1)
    return dict(ceiling_pct_acceptable=inband.mean(), null_median_abs_ln_oe=np.median(np.abs(np.log(np.maximum(O, .5) / E)), 1).mean())

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty(len(p)); run = 0
    for i, j in enumerate(o):
        run = max(run, min(1.0, (len(p) - i) * p[j])); adj[j] = run
    return adj
print('helpers ready')

## A-D. New strategies in the primary 2014 -> 2015 comparison

In [ ]:
G = by_hosp(upd, 'p_lgbm'); E_ = by_hosp(ev, 'p_lgbm')
UPDH = sorted(G)
rng = np.random.default_rng(13)
NET = tu.cached('net_all', network_fit, G, UPDH, True, True, True, None, rng)
def em_all():
    return {h: em_offset(tu.apply_logistic(G[h][0], NET['ag'], NET['bg']), NET['pi_net'], nboot=B_EM,
                         rng=np.random.default_rng(int(h))) for h in UPDH}
EMALL = tu.cached('em_all', em_all)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in NET.items() if k in ('ag', 'bg', 'mu', 'tau2', 'tau2_res', 'sigma_b2', 'pi_net')})
evs = evr.copy()
for c in ['S6_prev', 'S6_em', 'S3_cov', 'S3_cov_loho', 'S3_ss', 'S3_slope']:
    evs[c] = np.nan
# (A) label-free baselines, raw model, every eval hospital with update data
lg = lambda x: np.log(x / (1 - x))
for h in evs['hospital_id'].unique():
    if h not in G:
        continue
    msk = (evs['hospital_id'] == h).values; p = evs.loc[msk, 'p_lgbm'].values
    rate = np.clip(G[h][1].mean(), 0.5 / len(G[h][1]), 1 - 0.5 / len(G[h][1]))
    evs.loc[msk, 'S6_prev'] = tu.expit(tu.logit(p) + lg(rate) - lg(PI_SRC))
    evs.loc[msk, 'S6_em'] = tu.expit(tu.logit(p) + em_offset(G[h][0], PI_SRC)[0])
# (B) covariate EB, all-network fit
for h in evs['hospital_id'].unique():
    if h not in G:
        continue
    msk = (evs['hospital_id'] == h).values
    q = tu.apply_logistic(evs.loc[msk, 'p_lgbm'].values, NET['ag'], NET['bg'])
    a, s = tu.fit_intercept(tu.apply_logistic(G[h][0], NET['ag'], NET['bg']), G[h][1])
    evs.loc[msk, 'S3_cov'] = tu.expit(tu.logit(q) + posterior(cov_mean(NET, h), NET['tau2_res'], a, s))
# (B, C) leave-hospital-out covariate EB and semi-supervised (labels + EM + covariate prior)
def loho_fits():
    out = {}
    for i, h in enumerate(ELIG):
        others = [x for x in UPDH if x != h]
        out[h] = network_fit(G, others, True, True, True, None, np.random.default_rng(100 + i))
        if (i + 1) % 20 == 0:
            print(f'  LOHO network fits {i+1}/{len(ELIG)}', flush=True)
    return out
LOHO = tu.cached('loho_fits', loho_fits)
emsite = {}
for h in ELIG:
    msk = (evs['hospital_id'] == h).values
    F = LOHO[h]; q_u = tu.apply_logistic(G[h][0], F['ag'], F['bg'])
    a, s = tu.fit_intercept(q_u, G[h][1])
    eo, es = em_offset(q_u, F['pi_net'], nboot=B_EM, rng=np.random.default_rng(int(h)))
    emsite[h] = (eo, es)
    q = tu.apply_logistic(evs.loc[msk, 'p_lgbm'].values, F['ag'], F['bg'])
    mh = cov_mean(F, h)
    evs.loc[msk, 'S3_cov_loho'] = tu.expit(tu.logit(q) + posterior(mh, F['tau2_res'], a, s))
    evs.loc[msk, 'S3_ss'] = tu.expit(tu.logit(q) + posterior(mh, F['tau2_res'], a, s, eo, es, F['sigma_b2']))
# (D) bivariate intercept-slope EB after the global step
th, V = {}, {}
for h in UPDH:
    q = tu.apply_logistic(G[h][0], NET['ag'], NET['bg']); y = G[h][1]
    if y.sum() >= 3 and (len(y) - y.sum()) >= 3:
        beta, cov = tu.irls(np.column_stack([np.ones(len(y)), tu.logit(q)]), y)
        if np.all(np.isfinite(beta)) and np.all(np.isfinite(cov)):
            th[h], V[h] = np.asarray(beta), np.asarray(cov)
T = np.array([th[h] for h in th]); VV = np.array([V[h] for h in th])
Sig = np.cov(T.T) - VV.mean(0)
w_, U_ = np.linalg.eigh(Sig); Sig = U_ @ np.diag(np.maximum(w_, 1e-4)) @ U_.T
Wt = np.array([np.linalg.inv(Sig + v) for v in VV])
mu2 = np.linalg.solve(Wt.sum(0), np.einsum('kij,kj->i', Wt, T))
for h in evs['hospital_id'].unique():
    if h not in G:
        continue
    msk = (evs['hospital_id'] == h).values
    lq = tu.logit(tu.apply_logistic(evs.loc[msk, 'p_lgbm'].values, NET['ag'], NET['bg']))
    t_ = mu2 + Sig @ np.linalg.solve(Sig + V[h], th[h] - mu2) if h in th else mu2
    evs.loc[msk, 'S3_slope'] = tu.expit(t_[0] + t_[1] * lq)
tu.save_table(pd.DataFrame([dict(mu_a=mu2[0], mu_b=mu2[1], sd_a=np.sqrt(Sig[0, 0]), sd_b=np.sqrt(Sig[1, 1]),
                                 corr=Sig[0, 1] / np.sqrt(Sig[0, 0] * Sig[1, 1]), sites=len(th))]).round(4), 'D_bivariate_params')
tu.save_table(pd.DataFrame([dict(tau_plain=np.sqrt(NET['tau2']), tau_residual_cov=np.sqrt(NET['tau2_res']),
                                 sigma_b=np.sqrt(NET['sigma_b2']), pi_net=NET['pi_net'], pi_source=PI_SRC)]).round(4), 'B_network_params')
tu.save_table(pd.DataFrame([dict(term=c, gamma=g) for c, g in zip(np.array(NET['zcols'])[NET['keep']], NET['gamma'])]).round(4), 'B_cov_gamma')
evs.to_parquet(os.path.join(tu.MODELS, 'pred_eval_strategies_rev2.parquet'), index=False)

## E. Summary of all strategies on the 97 eligible hospitals

In [ ]:
ALL = ['S0_none', 'S1b_global_logistic', 'S2a_local_intercept', 'S2b_local_logistic', 'S3_eb', 'S3_eb_loho',
       'S3_glmm', 'S3_cov', 'S3_cov_loho', 'S3_ss', 'S3_slope', 'S6_prev', 'S6_em', 'S5_refit', 'S4_apache']
ALL = [s for s in ALL if s in evs]
phs, rows = {}, []
for s in ALL:
    d = evs if s != 'S4_apache' else evs[evs['S4_apache'].notna()]
    phs[s], r = hosp_summary(d, s, L[s], ELIG)
    r.update(tu.cached(f'ceil_{s}', chance_ceiling, d, s, ELIG)); r['key'] = s
    rows.append(r)
est = pd.DataFrame({s: phs[s].set_index('hospital_id')['ln_oe'] for s in ALL})
se_ = pd.DataFrame({s: phs[s].set_index('hospital_id')['ln_oe_se'] for s in ALL})
rr = np.random.default_rng(42); H = est.index.values; bt = {s: [] for s in ALL}
for b in range(B):
    idx = rr.choice(H, len(H), replace=True)
    for s in ALL:
        bt[s].append(tu.meta_analysis(est.loc[idx, s], se_.loc[idx, s], hksj=False)['tau'])
for r in rows:
    bb = np.array(bt[r['key']]); r['tau_lo'], r['tau_hi'] = np.nanpercentile(bb, 2.5), np.nanpercentile(bb, 97.5)
summ = pd.DataFrame(rows)
tu.save_table(summ.round(4), 'E_strategies_all')
diff = []
for ref in ['S3_eb', 'S3_cov_loho']:
    for s in [x for x in ALL if x not in (ref, 'S4_apache')]:
        t0 = summ.set_index('key').loc[s, 'tau'] - summ.set_index('key').loc[ref, 'tau']
        db = np.array(bt[s]) - np.array(bt[ref])
        diff.append(dict(reference=L[ref], other=L[s], tau_other_minus_ref=t0, lo=np.nanpercentile(db, 2.5),
                         hi=np.nanpercentile(db, 97.5)))
tu.save_table(pd.DataFrame(diff).round(4), 'E_tau_paired_differences')
a = est.abs(); wx = []
for ref in ['S3_eb', 'S3_cov_loho']:
    fam = []
    for s in [x for x in ALL if x != ref]:
        d = a[[ref, s]].dropna()
        fam.append(dict(reference=L[ref], comparison=L[s], k=len(d), median_ref=d[ref].median(), median_other=d[s].median(),
                        pct_ref_better=100 * (d[ref] < d[s]).mean(), p=wilcoxon(d[ref], d[s]).pvalue))
    fam = pd.DataFrame(fam); fam['p_holm'] = holm(fam['p'].values); wx.append(fam)
tu.save_table(pd.concat(wx).round(5), 'E_wilcoxon_holm')
pool = []
for s in ['S3_cov', 'S3_cov_loho', 'S3_ss', 'S3_slope', 'S6_prev', 'S6_em']:
    ci, _ = tu.bootstrap(evs['mortality'].values, evs[s].fillna(evs['S3_eb']).values, clusters=evs['hospital_id'].values,
                         B=tu.n_boot(500), metrics=['auroc', 'brier', 'oe', 'citl', 'slope', 'ici'])
    for _, r in ci.iterrows():
        pool.append(dict(strategy=L[s], metric=r['metric'], estimate=r['estimate'], lo=r['lo'], hi=r['hi']))
tu.save_table(pd.DataFrame(pool).round(4), 'E_pooled_new')

## F. Leave-region-out for the new methods (covariate EB without region covariates)

In [ ]:
reg = hd.set_index('hospital_id')['region'].fillna('Unknown')
def run_lro():
    out = []
    for R_ in ['Midwest', 'Northeast', 'South', 'West']:
        held = [h for h in ELIG if reg.get(h) == R_]
        if not held:
            continue
        others = [h for h in UPDH if reg.get(h) != R_]
        F = network_fit(G, others, True, False, True, None, np.random.default_rng(7))
        for h in held:
            q_u = tu.apply_logistic(G[h][0], F['ag'], F['bg']); a, s = tu.fit_intercept(q_u, G[h][1])
            eo, es = em_offset(q_u, F['pi_net'], nboot=B_EM, rng=np.random.default_rng(int(h)))
            pe, ye = E_[h]; q = tu.apply_logistic(pe, F['ag'], F['bg']); mh = cov_mean(F, h)
            for name, off in [('EB, other regions', posterior(F['mu'], F['tau2'], a, s)),
                              ('Covariate EB, other regions', posterior(mh, F['tau2_res'], a, s)),
                              ('Semi-supervised covariate EB, other regions', posterior(mh, F['tau2_res'], a, s, eo, es, F['sigma_b2'])),
                              ('EM label-shift only, other regions', eo), ('No updating', None)]:
                pp = pe if off is None else tu.expit(tu.logit(q) + off)
                o, ex = ye.sum(), pp.sum()
                out.append(dict(region=R_, hospital_id=h, method=name, oe=o / ex, ln_oe=np.log(o / ex),
                                ln_oe_se=np.sqrt((1 - o / len(ye)) / o)))
    return pd.DataFrame(out)
lro = tu.cached('lro', run_lro)
rows = []
for k, g_ in lro.groupby('method'):
    ma = tu.meta_analysis(g_['ln_oe'], g_['ln_oe_se'])
    rows.append(dict(method=k, hospitals=len(g_), pct_acceptable=100 * g_['oe'].between(*tu.OE_ACCEPT).mean(),
                     median_abs_ln_oe=g_['ln_oe'].abs().median(), tau=ma['tau']))
tu.save_table(pd.DataFrame(rows).round(4), 'F_leave_region_out')

## G. Label budget from 0 to 400 labels (all eligible hospitals; squared-error outputs for the risk analysis)

In [ ]:
NS = [0, 25, 50, 100, 200, 400]
nupd = {h: len(G[h][1]) for h in ELIG}
def lb_repeat(r):
    rng_ = np.random.default_rng(9000 + r); rows_ = []
    for h in ELIG:
        F = LOHO[h]; pu, yu = G[h]; pe, ye = E_[h]
        q_e = tu.apply_logistic(pe, F['ag'], F['bg']); mh = cov_mean(F, h); eo, es = emsite[h]
        o = ye.sum(); noise = (1 - o / len(ye)) / o
        for n in NS:
            if nupd[h] < n:
                continue
            preds = {}
            if n == 0:
                a = s = None
            else:
                ii = rng_.choice(len(yu), n, replace=False)
                a, s = tu.fit_intercept(tu.apply_logistic(pu[ii], F['ag'], F['bg']), yu[ii])
                al, _ = tu.fit_intercept(pu[ii], yu[ii]); preds['Local intercept'] = tu.apply_logistic(pe, al)
            preds['EB'] = tu.expit(tu.logit(q_e) + posterior(F['mu'], F['tau2'], a, s))
            preds['Covariate EB'] = tu.expit(tu.logit(q_e) + posterior(mh, F['tau2_res'], a, s))
            preds['Semi-supervised covariate EB'] = tu.expit(tu.logit(q_e) + posterior(mh, F['tau2_res'], a, s, eo, es, F['sigma_b2']))
            for k, pp in preds.items():
                ex = pp.sum(); l_ = np.log(o / ex)
                rows_.append(dict(rep=r, n=n, method=k, hospital_id=h, oe=o / ex, abs_ln_oe=abs(l_), sq_ln_oe=l_ ** 2, eval_noise=noise))
    return pd.DataFrame(rows_)
lb = pd.concat([tu.cached(f'lb_{r:03d}', lb_repeat, r) for r in range(R_LB)], ignore_index=True)
base = []
for h in ELIG:
    F = LOHO[h]; pe, ye = E_[h]; o = ye.sum(); noise = (1 - o / len(ye)) / o
    rate = np.clip(G[h][1].mean(), 0.5 / len(G[h][1]), 1 - 0.5 / len(G[h][1]))
    q_e = tu.apply_logistic(pe, F['ag'], F['bg'])
    for k, pp in [('No updating', pe), ('Global logistic (LOHO)', q_e),
                  ('Prevalence correction (site rate)', tu.expit(tu.logit(pe) + lg(rate) - lg(PI_SRC))),
                  ('EM label-shift (no labels)', tu.expit(tu.logit(q_e) + emsite[h][0]))]:
        l_ = np.log(o / pp.sum())
        base.append(dict(hospital_id=h, method=k, oe=o / pp.sum(), abs_ln_oe=abs(l_), sq_ln_oe=l_ ** 2, eval_noise=noise))
base = pd.DataFrame(base)
FIX35 = [h for h in ELIG if nupd[h] >= 400]
out = []
for setname, fn in [('all', lambda n: [h for h in ELIG if nupd[h] >= max(n, 0)]), ('fixed35', lambda n: FIX35)]:
    for n in NS:
        hs = fn(n)
        for src in (lb[(lb['n'] == n) & lb['hospital_id'].isin(hs)], None):
            d = src if src is not None else base[base['hospital_id'].isin(hs)]
            for k, g_ in d.groupby('method'):
                out.append(dict(hospital_set=setname, n=n, hospitals=len(hs), method=k,
                                pct_acceptable=100 * g_['oe'].between(*tu.OE_ACCEPT).mean(),
                                median_abs_ln_oe=g_['abs_ln_oe'].median(), mean_sq_ln_oe=g_['sq_ln_oe'].mean(),
                                mean_eval_noise=g_['eval_noise'].mean()))
lbt = pd.DataFrame(out)
tu.save_table(lbt.round(4), 'G_label_budget')

## H. Inputs for the closed-form risk analysis (per eligible hospital)

In [ ]:
rows = []
for h in ELIG:
    q14 = tu.apply_logistic(G[h][0], NET['ag'], NET['bg']); q15 = tu.apply_logistic(E_[h][0], NET['ag'], NET['bg'])
    a14, s14 = tu.fit_intercept(q14, G[h][1]); a15, s15 = tu.fit_intercept(q15, E_[h][1])
    rows.append(dict(hospital_id=h, n14=len(q14), n15=len(q15), info14=float(np.mean(q14 * (1 - q14))),
                     info15=float(np.mean(q15 * (1 - q15))), kappa15=float(np.sum(q15 * (1 - q15)) / np.sum(q15)),
                     o15=float(E_[h][1].sum()), a14=a14, s14=s14, a15=a15, s15=s15,
                     m_cov=cov_mean(NET, h), large=nupd[h] >= 400))
th_in = pd.DataFrame(rows)
d_ = th_in['a15'] - th_in['a14']
omega2 = max(0.0, float(np.var(d_, ddof=1) - np.mean(th_in['s14'] ** 2 + th_in['s15'] ** 2)))
ma14 = tu.meta_analysis(th_in['a14'], th_in['s14'], hksj=False)
g_, t2r, _ = meta_reg_mom(th_in['a14'].values, th_in['s14'].values, zmatrix(list(th_in['hospital_id']), True))
tu.save_table(th_in.round(5), 'H_theory_sites', show=False)
tu.save_table(pd.DataFrame([dict(mu_elig=ma14['mu'], tau2_elig=ma14['tau2'], tau2_res_elig=t2r, omega2_drift=omega2,
                                 mu_net=NET['mu'], tau2_net=NET['tau2'], tau2_res_net=NET['tau2_res'])]).round(5), 'H_theory_params')

## I. MLP base model (sensitivity: method does not depend on the base learner)

In [ ]:
dev = m[m['period'] == 'dev']; tt = m[m['period'] != 'dev']
def fit_mlp():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('mlp', MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-3, early_stopping=True, validation_fraction=0.15,
                                           max_iter=30 if tu.FAST else 300, random_state=tu.SEED))]).fit(dev[FEATS].astype(float), dev['mortality'].astype(int))
mlp = tu.cached('mlp', fit_mlp)
pe_all = pd.Series(mlp.predict_proba(e.set_index('stay_id').loc[pred['stay_id'], FEATS].astype(float))[:, 1], index=pred.index)
pp = pred.copy(); pp['p_mlp'] = pe_all.values
auc_tt = tu.roc_auc_score(tt['mortality'], mlp.predict_proba(tt[FEATS].astype(float))[:, 1])
auc_e = tu.roc_auc_score(pp['mortality'], pp['p_mlp'])
def strategies_for(dfu, dfe, pcol, ycol, label, pi_src):
    """S0, S1b, S2a, S3, covariate EB (all-network), prevalence and EM corrections, on hospitals with >=100 eval patients,
    >=10 events and >=50 update patients."""
    evs_, info, _ = tu.run_strategies(dfu, dfe, pcol, ycol=ycol, apache_col='__none__')
    Gx = {h: (d[pcol].values.astype(float), d[ycol].values.astype(float)) for h, d in dfu.groupby('hospital_id')}
    F = network_fit(Gx, sorted(Gx), True, True, False)
    for c in ['S3_cov', 'S6_prev', 'S6_em']:
        evs_[c] = np.nan
    for h in evs_['hospital_id'].unique():
        if h not in Gx:
            continue
        msk = (evs_['hospital_id'] == h).values; p = evs_.loc[msk, pcol].values
        q = tu.apply_logistic(p, F['ag'], F['bg']); a, s = tu.fit_intercept(tu.apply_logistic(Gx[h][0], F['ag'], F['bg']), Gx[h][1])
        evs_.loc[msk, 'S3_cov'] = tu.expit(tu.logit(q) + posterior(cov_mean(F, h), F['tau2_res'], a, s))
        rate = np.clip(Gx[h][1].mean(), 0.5 / len(Gx[h][1]), 1 - 0.5 / len(Gx[h][1]))
        evs_.loc[msk, 'S6_prev'] = tu.expit(tu.logit(p) + lg(rate) - lg(pi_src))
        evs_.loc[msk, 'S6_em'] = tu.expit(tu.logit(p) + em_offset(Gx[h][0], pi_src)[0])
    c_ = evs_.groupby('hospital_id').agg(n=(ycol, 'size'), d=(ycol, 'sum'))
    nu = dfu.groupby('hospital_id').size()
    el = [h for h in c_[(c_.n >= tu.MIN_HOSP_N) & (c_.d >= tu.MIN_HOSP_EVENTS)].index if nu.get(h, 0) >= tu.MIN_UPDATE_N]
    out, est_ = [], {}
    for s in ['S0_none', 'S1b_global_logistic', 'S2a_local_intercept', 'S3_eb', 'S3_cov', 'S6_prev', 'S6_em']:
        ph, r = hosp_summary(evs_, s, L[s], el, ycol=ycol); r['analysis'] = label; r['key'] = s
        r['ceiling_pct_acceptable'] = chance_ceiling(evs_, s, el, R=min(R_SIM, 500))['ceiling_pct_acceptable']
        out.append(r); est_[s] = ph.set_index('hospital_id')['ln_oe'].abs()
    o = pd.DataFrame(out); A_ = pd.DataFrame(est_)
    for ref in ['S3_eb', 'S3_cov']:
        ps = [np.nan if s == ref else wilcoxon(A_[ref], A_[s]).pvalue for s in o['key']]
        mask = ~np.isnan(ps); adj = np.full(len(ps), np.nan); adj[mask] = holm(np.array(ps)[mask])
        o[f'p_vs_{ref}'] = ps; o[f'p_holm_vs_{ref}'] = adj
    return o
mlp_tab = strategies_for(pp[pp['split'] == 'update'].reset_index(drop=True), pp[pp['split'] == 'eval'].reset_index(drop=True),
                         'p_mlp', 'mortality', 'Base model: MLP', PI_SRC)
mlp_tab['auroc_mimic_temporal'] = auc_tt; mlp_tab['auroc_eicu'] = auc_e
tu.save_table(mlp_tab.round(4), 'I_mlp_strategies')

## J. Second outcome (ICU stay >= 7 days) with the new methods and Holm adjustment

In [ ]:
def raw(name):
    return pd.read_parquet(os.path.join(tu.DATA_RAW, name + '.parquet'))
mc = raw('mimic_cohort')[['stay_id', 'icu_los_h']].apply(pd.to_numeric, errors='coerce').drop_duplicates('stay_id')
ec = raw('eicu_cohort')[['stay_id', 'unitdischargeoffset']].apply(pd.to_numeric, errors='coerce').drop_duplicates('stay_id')
m2 = m.merge(mc, on='stay_id', how='left'); e2 = e.merge(ec, on='stay_id', how='left')
m2['prolonged'] = (m2['icu_los_h'] >= 168).astype(int); e2['prolonged'] = (e2['unitdischargeoffset'] >= 7 * 1440).astype(int)
dev2 = m2[m2['period'] == 'dev']
def fit_los():
    return lgb.LGBMClassifier(n_estimators=bp['n_estimators'], subsample_freq=1, random_state=tu.SEED, verbose=-1,
                              n_jobs=-1, **bp['params']).fit(dev2[FEATS].astype(float), dev2['prolonged'].astype(int))
mdl = tu.cached('los_model', fit_los)
e2['p_los'] = mdl.predict_proba(e2[FEATS].astype(float))[:, 1]
pl = e2[['stay_id', 'hospital_id', 'prolonged', 'p_los', 'age']].merge(pred[['stay_id', 'split']], on='stay_id')
los_tab = strategies_for(pl[pl['split'] == 'update'].reset_index(drop=True), pl[pl['split'] == 'eval'].reset_index(drop=True),
                         'p_los', 'prolonged', 'ICU stay >= 7 d', float(dev2['prolonged'].mean()))
tu.save_table(los_tab.round(4), 'J_los_strategies')

In [ ]:
tu.finish_notebook({'strategies': summ.drop(columns=['key']).round(4).to_dict('records'),
                    'network': {k: (float(v) if isinstance(v, (float, np.floating)) else None) for k, v in NET.items() if k in ('mu', 'tau2', 'tau2_res', 'sigma_b2', 'pi_net')},
                    'mlp': mlp_tab.round(4).to_dict('records'), 'los': los_tab.round(4).to_dict('records')})